# RAG (Retrieval-Augmented Generation) — Step-by-Step Guide

**What is RAG?**  
Large Language Models (LLMs) are trained on general internet corpora and have a fixed knowledge cutoff date.  
They cannot answer questions about **your private documents** — enterprise policies, product guides, customer support runbooks, or internal technical manuals.

**RAG bridges this gap** through a two-phase architecture:
1. **Retrieve** — Locate the top-k most relevant passages from your indexed knowledge base for any given user inquiry.
2. **Generate** — Inject those specific passages as grounded context into the LLM prompt, instructing the model to synthesize an accurate answer strictly from the provided facts.

---

### Complete RAG System Architecture

```
┌───────────────────────────────┐
│     telecom_guide.pdf         │  Source Document
└───────────────┬───────────────┘
                │  Step 1: PyPDFLoader
                ▼
┌───────────────────────────────┐
│        9 PDF Pages            │  Raw Page Documents
└───────────────┬───────────────┘
                │  Step 2: RecursiveCharacterTextSplitter (chunk_size=600, overlap=100)
                ▼
┌───────────────────────────────┐
│         37 Chunks             │  Granular Context Chunks
└───────────────┬───────────────┘
                │  Step 3: HuggingFaceEmbeddings (all-MiniLM-L6-v2)
                ▼
┌───────────────────────────────┐     similarity_search(k=3)
│       Chroma Vector DB        │ ──────────────────────────────┐
│         (in-memory)           │                               │
└───────────────▲───────────────┘                               ▼
                │                                    ┌─────────────────────┐
                │ embed query                        │ Top-3 Context Docs  │
                │                                    └──────────┬──────────┘
┌───────────────┴───────────────┐                               │
│          User Query           │                               │ format_docs()
│   "How does international     │                               ▼
│    roaming work?"             │ ────────────────────► ┌──────────────────┐
└───────────────────────────────┘                       │ Grounded Context │
                                                        └───────┬──────────┘
                                                                │
                                                                ▼
                                                        ┌──────────────────┐
                                                        │ ChatPromptTemplate│
                                                        └───────┬──────────┘
                                                                │
                                                                ▼
                                                        ┌──────────────────┐
                                                        │    ChatOpenAI    │
                                                        │ (or Gemini/Groq) │
                                                        └───────┬──────────┘
                                                                │
                                                                ▼
                                                        ┌──────────────────┐
                                                        │ Synthesized Answer│
                                                        └──────────────────┘
```

**Document Used:** `telecom_guide.pdf` (Technical Reference Guide covering 2G/3G/4G/5G, VoLTE, roaming, and billing).


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()


---
## Step 1 — Load Document (PyPDFLoader)

`PyPDFLoader` reads each page of the PDF and returns a list of LangChain `Document` objects.  
Each `Document` contains `.page_content` (the extracted text) and `.metadata` (page number, source path).


In [2]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF.")
print("\n--- First page preview (first 500 chars) ---")
print(pages[0].page_content[:500])


---
## Step 2 — Split into Semantic Chunks

**Why Chunk?**
- Embedding models have token context limits (e.g., 512 tokens for `all-MiniLM-L6-v2`).
- Retrieval precision is higher when searching specific paragraphs rather than entire multi-page documents.

**Key Parameters:**
- `chunk_size = 600`: Max characters per chunk (~150 words).
- `chunk_overlap = 100`: Shared text boundary between adjacent chunks to preserve contextual continuity.


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,       # ~150 words per chunk
    chunk_overlap=100,    # overlap keeps context at boundaries
    separators=["\n\n", "\n", ".", " "],  # tries paragraph -> line -> sentence -> words
)

chunks = splitter.split_documents(pages)
print(f"Total chunks created: {len(chunks)}")


---
## Step 3 — Embed Chunks and Index in ChromaDB

**What is an Embedding?**  
A dense vector (numerical array) representing the semantic meaning of the text. Similar texts have high cosine similarity.

Here we use:
- **Embedding Model:** `sentence-transformers/all-MiniLM-L6-v2` (fast, lightweight, 384-dimensional vectors).
- **Vector Store:** `Chroma` with an in-memory client (`chromadb.EphemeralClient()`) to ensure clean idempotency on every notebook run.


In [4]:
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("Loading embedding model (downloads ~90 MB on first run)...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Embedding all chunks and storing in Chroma (in memory)...")
# Use fresh EphemeralClient on every run to prevent duplicate accumulation
client = chromadb.EphemeralClient()
vector_store = Chroma.from_documents(chunks, embeddings, client=client)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored.")


---
## Step 4 — Test Vector Retriever (Top-K Semantic Search)

Before connecting the LLM, we verify that the retriever correctly pulls the top `k=3` relevant chunks for a domain question.


In [5]:
# k must be an integer
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "What is VoLTE and how does it improve call quality"
retrieved = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Retrieved {len(retrieved)} chunks:\n")
for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()


---
## Step 5 — Assemble the End-to-End RAG Chain (LCEL)

We connect the components using **LangChain Expression Language (LCEL)**:

```
Question ────────┬──────────────────────────────────────────┐
                 ▼                                          ▼
             Retriever                               RunnablePassthrough()
                 │                                          │
                 ▼                                          │
            format_docs()                                   │
                 │                                          │
                 ▼ (context)                                ▼ (question)
          ┌─────────────────────────────────────────────────────┐
          │                  ChatPromptTemplate                 │
          └──────────────────────────┬──────────────────────────┘
                                     │
                                     ▼
          ┌─────────────────────────────────────────────────────┐
          │                       Chat Model                    │
          └──────────────────────────┬──────────────────────────┘
                                     │
                                     ▼
          ┌─────────────────────────────────────────────────────┐
          │                    StrOutputParser()                │
          └──────────────────────────┬──────────────────────────┘
                                     │
                                     ▼
                               Final Answer
```


In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

# --- Helper: join retrieved chunks into a single context string ---
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

# --- System prompt: ground the LLM strictly in the retrieved context ---
SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

# --- LLM via OpenAI API (or swap with ChatGoogleGenerativeAI) ---
llm = ChatOpenAI(
    model="gpt-6-luna",
    temperature=0,
)

# --- Assemble the LCEL Chain ---
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled.")

question = "How does international roaming work and what charges should I expect?"
print(f"Q: {question}\n")
print("A:", chain.invoke(question))
